# Seeded Run 1

This notebook trains five matched seeded pairs for the 2-class SHD odd/even parity task.

Each training cell handles one seed pair independently. Once a pair finishes, its checkpoints and summaries are already saved, so you can stop there and later continue with the next seed cell instead of rerunning a full loop.

Cell 2 defines the seeds used in this notebook. Cell 3 verifies three things before training starts:
1. this notebook is locked to the 2-class odd/even classification task
2. the homogeneous and heterogeneous model definitions are set up correctly
3. the log-normal hidden tau_m samples vary across the configured seeds

In [5]:
from pathlib import Path
import importlib

import pandas as pd
from IPython.display import display

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
DEVICE = seeded_runs_common.DEVICE
SHD_TEST = seeded_runs_common.SHD_TEST
SHD_TRAIN = seeded_runs_common.SHD_TRAIN
TAU_ARTIFACT_PATH = seeded_runs_common.TAU_ARTIFACT_PATH
TASKS = seeded_runs_common.TASKS
build_pair_summary_row = seeded_runs_common.build_pair_summary_row
build_sampling_preview_rows = seeded_runs_common.build_sampling_preview_rows
build_seeded_pair = seeded_runs_common.build_seeded_pair
expected_2class_odd_even_map = seeded_runs_common.expected_2class_odd_even_map
load_default_caches = seeded_runs_common.load_default_caches
read_manifest_rows = seeded_runs_common.read_manifest_rows
run_pair_training = seeded_runs_common.run_pair_training
upsert_rows = seeded_runs_common.upsert_rows
write_manifest_rows = seeded_runs_common.write_manifest_rows

RUN_LABEL = "seeded_run_1"
TASK_KEY = "2class"
MEM_DISTRIBUTION_FAMILY = "lognormal"
MASTER_SEEDS = [101, 202, 210, 340, 440]

RUN_DIR = CHECKPOINT_ROOT / RUN_LABEL / f"{TASK_KEY}_{MEM_DISTRIBUTION_FAMILY}"
RESULT_STEM = RUN_DIR / f"{RUN_LABEL}_checkpoint_summary"
PAIR_STEM = RUN_DIR / f"{RUN_LABEL}_pair_summary"
PAIRED_ACC_CSV = RUN_DIR / f"{RUN_LABEL}_paired_accuracy.csv"

TASK_DEF = TASKS[TASK_KEY]
EXPECTED_LABEL_MAP = expected_2class_odd_even_map()

print(f"Device: {DEVICE}")
print(f"Train file exists: {SHD_TRAIN.exists()}")
print(f"Test file exists: {SHD_TEST.exists()}")
print(f"Tau artifact exists: {TAU_ARTIFACT_PATH.exists()}")
print(f"Run directory: {RUN_DIR}")
print(f"Master seeds: {MASTER_SEEDS}")
print(f"Task name: {TASK_DEF['task_name']}")
print(f"Task outputs: {TASK_DEF['nb_outputs']}")

Device: cuda
Train file exists: True
Test file exists: True
Tau artifact exists: True
Run directory: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns\seeded_run_1\2class_lognormal
Master seeds: [101, 202, 210, 340, 440]
Task name: binary_parity
Task outputs: 2


In [6]:
assert TASK_KEY == "2class"
assert TASK_DEF["nb_outputs"] == 2
assert TASK_DEF["task_name"] == "binary_parity"
assert TASK_DEF["task_label_map"] == EXPECTED_LABEL_MAP

preview = build_seeded_pair(
    master_seed=MASTER_SEEDS[0],
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
preview_meta = preview["metadata"]

assert preview["hom_prms"]["nb_outputs"] == 2
assert preview["hetero_prms"]["nb_outputs"] == 2
assert not preview["hom_model"].network[0].alpha.requires_grad
assert not preview["hom_model"].network[0].beta.requires_grad
assert not preview["hetero_model"].network[0].alpha.requires_grad
assert not preview["hetero_model"].network[0].beta.requires_grad
assert preview_meta["linear_sync_verified"]
assert preview_meta["hom_hidden_tau_unique"] == 1
assert preview_meta["hetero_hidden_tau_unique"] > 1

sampling_rows = build_sampling_preview_rows(
    MASTER_SEEDS,
    task_key=TASK_KEY,
    mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
)
sampling_df = pd.DataFrame(sampling_rows).sort_values("pair_seed").reset_index(drop=True)

assert not sampling_df["sample_matches_previous"].any()

display(sampling_df)
print("2-class odd/even task mapping verified.")
print("Homogeneous anchor and heterogeneous sampled definitions verified.")
print("Log-normal hidden tau_m sampling varies across the configured seeds.")

,pair_seed,task_key,task_name,mem_distribution_family,linear_sync_verified,hetero_hidden_tau_unique,hom_hidden_tau_unique,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,sample_matches_previous
0,101,2class,binary_parity,lognormal,True,30,1,24.687252,3.410472,99.749887,37.499115,30.195999,False
1,202,2class,binary_parity,lognormal,True,28,1,29.308363,6.953492,99.749887,39.884979,30.480470,False
2,210,2class,binary_parity,lognormal,True,31,1,22.405558,3.855315,99.749887,34.397371,29.713145,False
3,340,2class,binary_parity,lognormal,True,31,1,23.273864,4.656051,99.749887,35.008043,30.389582,False
4,440,2class,binary_parity,lognormal,True,28,1,28.874491,5.547958,99.749887,41.448743,31.915950,False


2-class odd/even task mapping verified.
Homogeneous anchor and heterogeneous sampled definitions verified.
Log-normal hidden tau_m sampling varies across the configured seeds.


In [9]:
train_cache, test_cache = load_default_caches()

Pre-loading SHD training data into RAM...
  SHDCache: 8156 samples loaded from shd_train.h5
Pre-loading SHD test data into RAM...
  SHDCache: 2264 samples loaded from shd_test.h5


In [7]:
CHECKPOINT_KEY_FIELDS = ["pair_seed", "pair_role"]
PAIR_KEY_FIELDS = ["pair_seed"]

def show_run_status():
    checkpoint_rows = read_manifest_rows(RESULT_STEM)
    pair_rows = read_manifest_rows(PAIR_STEM)

    checkpoint_df = pd.DataFrame(checkpoint_rows)
    pair_df = pd.DataFrame(pair_rows)

    if not checkpoint_df.empty:
        checkpoint_df = checkpoint_df.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    if not pair_df.empty:
        pair_df = pair_df.sort_values("pair_seed").reset_index(drop=True)

    paired_acc_df = pd.DataFrame()
    if not checkpoint_df.empty:
        paired_acc_df = (
            checkpoint_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
            .reset_index()
            .sort_values("pair_seed")
            .reset_index(drop=True)
        )
        if {"heterogeneous_sampled", "homogeneous_anchor"}.issubset(paired_acc_df.columns):
            paired_acc_df["hetero_minus_hom"] = (
                paired_acc_df["heterogeneous_sampled"] - paired_acc_df["homogeneous_anchor"]
            )
            paired_acc_df.to_csv(PAIRED_ACC_CSV, index=False)

    return pair_df, checkpoint_df, paired_acc_df

def train_one_seed(master_seed):
    run_rows, pair_meta = run_pair_training(
        master_seed=master_seed,
        train_cache=train_cache,
        test_cache=test_cache,
        task_key=TASK_KEY,
        mem_distribution_family=MEM_DISTRIBUTION_FAMILY,
        run_label=RUN_LABEL,
        skip_existing=True,
    )

    checkpoint_rows = upsert_rows(
        read_manifest_rows(RESULT_STEM),
        run_rows,
        CHECKPOINT_KEY_FIELDS,
    )
    pair_rows = upsert_rows(
        read_manifest_rows(PAIR_STEM),
        [build_pair_summary_row(pair_meta)],
        PAIR_KEY_FIELDS,
    )

    write_manifest_rows(checkpoint_rows, RESULT_STEM)
    write_manifest_rows(pair_rows, PAIR_STEM)

    pair_df, checkpoint_df, paired_acc_df = show_run_status()
    seed_df = checkpoint_df[checkpoint_df["pair_seed"] == master_seed].reset_index(drop=True)
    return seed_df, pair_df, checkpoint_df, paired_acc_df

print("Run helpers ready.")
print("Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.")

Run helpers ready.
Execute one seed cell at a time. Finished seeds are reused from saved checkpoints.


In [10]:
# Train or reuse seed pair 101
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[0])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[0]].reset_index(drop=True))


Seed 101 :: training homogeneous_anchor...
  epoch=001  train_acc=0.571  test_acc=0.625  (2.9 min)
  epoch=002  train_acc=0.682  test_acc=0.679  (2.6 min)
  epoch=003  train_acc=0.717  test_acc=0.749  (2.7 min)
  epoch=004  train_acc=0.755  test_acc=0.763  (2.1 min)
  epoch=005  train_acc=0.764  test_acc=0.766  (2.1 min)
  epoch=006  train_acc=0.783  test_acc=0.778  (2.1 min)
  epoch=007  train_acc=0.784  test_acc=0.772  (2.1 min)
  epoch=008  train_acc=0.797  test_acc=0.785  (2.3 min)
  epoch=009  train_acc=0.805  test_acc=0.751  (2.5 min)
  epoch=010  train_acc=0.820  test_acc=0.753  (2.8 min)
  epoch=011  train_acc=0.823  test_acc=0.769  (2.7 min)
  epoch=012  train_acc=0.836  test_acc=0.758  (2.7 min)
  epoch=013  train_acc=0.846  test_acc=0.798  (2.7 min)
  epoch=014  train_acc=0.851  test_acc=0.804  (2.7 min)
  epoch=015  train_acc=0.856  test_acc=0.755  (2.7 min)
  epoch=016  train_acc=0.870  test_acc=0.817  (2.8 min)
  epoch=017  train_acc=0.872  test_acc=0.734  (2.4 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_1,2class,binary_parity,101,heterogeneous_sampled,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,True,C:\Users\Priya\Desktop\research project (SNN I...,3253.226868,0.824205,0.461067
1,seeded_run_1,2class,binary_parity,101,homogeneous_anchor,lognormal,10.0,24.687252,24.687252,24.687252,24.687252,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3615.519982,0.826855,0.424700


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,101,2class,binary_parity,lognormal,10.0,24.687252,3.410472,99.749887,37.499115,30.195999,30,1,True,6


In [11]:
# Train or reuse seed pair 202
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[1])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[1]].reset_index(drop=True))


Seed 202 :: training homogeneous_anchor...
  epoch=001  train_acc=0.496  test_acc=0.505  (2.1 min)
  epoch=002  train_acc=0.548  test_acc=0.708  (2.1 min)
  epoch=003  train_acc=0.688  test_acc=0.625  (2.2 min)
  epoch=004  train_acc=0.716  test_acc=0.731  (2.8 min)
  epoch=005  train_acc=0.762  test_acc=0.667  (2.7 min)
  epoch=006  train_acc=0.792  test_acc=0.690  (2.8 min)
  epoch=007  train_acc=0.805  test_acc=0.712  (2.7 min)
  epoch=008  train_acc=0.813  test_acc=0.730  (2.7 min)
  epoch=009  train_acc=0.824  test_acc=0.742  (2.7 min)
  epoch=010  train_acc=0.824  test_acc=0.686  (2.8 min)
  epoch=011  train_acc=0.836  test_acc=0.757  (2.8 min)
  epoch=012  train_acc=0.843  test_acc=0.745  (2.9 min)
  epoch=013  train_acc=0.849  test_acc=0.714  (2.8 min)
  epoch=014  train_acc=0.856  test_acc=0.746  (2.8 min)
  epoch=015  train_acc=0.865  test_acc=0.743  (2.9 min)
  epoch=016  train_acc=0.867  test_acc=0.742  (2.8 min)
  epoch=017  train_acc=0.872  test_acc=0.756  (2.9 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_1,2class,binary_parity,202,heterogeneous_sampled,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.48047,True,C:\Users\Priya\Desktop\research project (SNN I...,4080.311442,0.845406,0.420902
1,seeded_run_1,2class,binary_parity,202,homogeneous_anchor,lognormal,10.0,29.308363,29.308363,29.308363,29.308363,0.00000,True,C:\Users\Priya\Desktop\research project (SNN I...,4147.010132,0.779594,0.510261


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,202,2class,binary_parity,lognormal,10.0,29.308363,6.953492,99.749887,39.884979,30.48047,28,1,True,6


In [12]:
# Train or reuse seed pair 210
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[2])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[2]].reset_index(drop=True))


Seed 210 :: training homogeneous_anchor...
  epoch=001  train_acc=0.546  test_acc=0.611  (2.8 min)
  epoch=002  train_acc=0.639  test_acc=0.700  (2.8 min)
  epoch=003  train_acc=0.681  test_acc=0.701  (2.8 min)
  epoch=004  train_acc=0.750  test_acc=0.767  (2.8 min)
  epoch=005  train_acc=0.774  test_acc=0.748  (2.8 min)
  epoch=006  train_acc=0.791  test_acc=0.685  (2.8 min)
  epoch=007  train_acc=0.805  test_acc=0.752  (2.8 min)
  epoch=008  train_acc=0.814  test_acc=0.772  (2.8 min)
  epoch=009  train_acc=0.826  test_acc=0.703  (2.8 min)
  epoch=010  train_acc=0.832  test_acc=0.759  (2.8 min)
  epoch=011  train_acc=0.841  test_acc=0.798  (2.5 min)
  epoch=012  train_acc=0.845  test_acc=0.754  (2.9 min)
  epoch=013  train_acc=0.861  test_acc=0.788  (2.3 min)
  epoch=014  train_acc=0.867  test_acc=0.771  (2.2 min)
  epoch=015  train_acc=0.867  test_acc=0.760  (2.5 min)
  epoch=016  train_acc=0.881  test_acc=0.780  (2.3 min)
  epoch=017  train_acc=0.892  test_acc=0.796  (2.1 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_1,2class,binary_parity,210,heterogeneous_sampled,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,True,C:\Users\Priya\Desktop\research project (SNN I...,3851.373862,0.801237,0.478489
1,seeded_run_1,2class,binary_parity,210,homogeneous_anchor,lognormal,10.0,22.405558,22.405558,22.405558,22.405558,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,3684.120756,0.730565,0.677380


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,210,2class,binary_parity,lognormal,10.0,22.405558,3.855315,99.749887,34.397371,29.713145,31,1,True,6


In [13]:
# Train or reuse seed pair 340
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[3])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[3]].reset_index(drop=True))


Seed 340 :: training homogeneous_anchor...
  epoch=001  train_acc=0.604  test_acc=0.646  (2.9 min)
  epoch=002  train_acc=0.677  test_acc=0.696  (2.7 min)
  epoch=003  train_acc=0.711  test_acc=0.653  (2.7 min)
  epoch=004  train_acc=0.744  test_acc=0.641  (2.7 min)
  epoch=005  train_acc=0.764  test_acc=0.639  (2.7 min)
  epoch=006  train_acc=0.763  test_acc=0.673  (2.7 min)
  epoch=007  train_acc=0.796  test_acc=0.666  (2.8 min)
  epoch=008  train_acc=0.802  test_acc=0.698  (2.8 min)
  epoch=009  train_acc=0.816  test_acc=0.707  (2.8 min)
  epoch=010  train_acc=0.834  test_acc=0.696  (2.7 min)
  epoch=011  train_acc=0.841  test_acc=0.727  (2.8 min)
  epoch=012  train_acc=0.855  test_acc=0.739  (3.0 min)
  epoch=013  train_acc=0.853  test_acc=0.696  (2.9 min)
  epoch=014  train_acc=0.870  test_acc=0.705  (2.8 min)
  epoch=015  train_acc=0.871  test_acc=0.735  (3.0 min)
  epoch=016  train_acc=0.881  test_acc=0.759  (3.3 min)
  epoch=017  train_acc=0.890  test_acc=0.727  (3.3 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_1,2class,binary_parity,340,heterogeneous_sampled,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,True,C:\Users\Priya\Desktop\research project (SNN I...,3794.156995,0.791519,0.514167
1,seeded_run_1,2class,binary_parity,340,homogeneous_anchor,lognormal,10.0,23.273864,23.273864,23.273864,23.273864,0.000000,True,C:\Users\Priya\Desktop\research project (SNN I...,4360.563598,0.753092,0.621278


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,340,2class,binary_parity,lognormal,10.0,23.273864,4.656051,99.749887,35.008043,30.389582,31,1,True,6


In [14]:
# Train or reuse seed pair 440
seed_df, pair_df, checkpoint_df, paired_acc_df = train_one_seed(MASTER_SEEDS[4])
display(seed_df)
display(pair_df[pair_df["pair_seed"] == MASTER_SEEDS[4]].reset_index(drop=True))


Seed 440 :: training homogeneous_anchor...
  epoch=001  train_acc=0.570  test_acc=0.612  (2.2 min)
  epoch=002  train_acc=0.634  test_acc=0.686  (2.2 min)
  epoch=003  train_acc=0.676  test_acc=0.655  (2.2 min)
  epoch=004  train_acc=0.703  test_acc=0.736  (2.2 min)
  epoch=005  train_acc=0.765  test_acc=0.664  (2.2 min)
  epoch=006  train_acc=0.789  test_acc=0.746  (2.2 min)
  epoch=007  train_acc=0.806  test_acc=0.733  (2.2 min)
  epoch=008  train_acc=0.816  test_acc=0.757  (2.2 min)
  epoch=009  train_acc=0.833  test_acc=0.813  (2.2 min)
  epoch=010  train_acc=0.845  test_acc=0.803  (2.2 min)
  epoch=011  train_acc=0.852  test_acc=0.787  (2.2 min)
  epoch=012  train_acc=0.870  test_acc=0.802  (2.2 min)
  epoch=013  train_acc=0.869  test_acc=0.758  (2.2 min)
  epoch=014  train_acc=0.879  test_acc=0.742  (2.2 min)
  epoch=015  train_acc=0.885  test_acc=0.795  (2.2 min)
  epoch=016  train_acc=0.904  test_acc=0.788  (2.2 min)
  epoch=017  train_acc=0.913  test_acc=0.809  (2.2 min)
  ep

,run_label,task_key,task_name,pair_seed,pair_role,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hidden_tau_mem_min_ms,hidden_tau_mem_max_ms,hidden_tau_mem_mean_ms,hidden_tau_mem_std_ms,linear_sync_verified,checkpoint,elapsed_s,final_test_acc,final_test_loss
0,seeded_run_1,2class,binary_parity,440,heterogeneous_sampled,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.91595,True,C:\Users\Priya\Desktop\research project (SNN I...,3227.709298,0.810512,0.484213
1,seeded_run_1,2class,binary_parity,440,homogeneous_anchor,lognormal,10.0,28.874491,28.874491,28.874491,28.874491,0.00000,True,C:\Users\Priya\Desktop\research project (SNN I...,3279.749818,0.848498,0.382974


,pair_seed,task_key,task_name,mem_distribution_family,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,hetero_hidden_tau_min_ms,hetero_hidden_tau_max_ms,hetero_hidden_tau_mean_ms,hetero_hidden_tau_std_ms,hetero_hidden_tau_unique,hom_hidden_tau_unique,linear_sync_verified,checked_parameter_count
0,440,2class,binary_parity,lognormal,10.0,28.874491,5.547958,99.749887,41.448743,31.91595,28,1,True,6


In [8]:
pair_df, checkpoint_df, paired_acc_df = show_run_status()

if pair_df.empty:
    print("No saved seed summaries yet.")
else:
    display(pair_df)

if checkpoint_df.empty:
    print("No saved checkpoint summaries yet.")
else:
    display(checkpoint_df)

if not paired_acc_df.empty:
    display(paired_acc_df)
    print(f"Saved paired accuracy view to: {PAIRED_ACC_CSV}")

No saved seed summaries yet.
No saved checkpoint summaries yet.
